# N-gram language model

In [ ]:
import numpy as np
import math

## Phase 1: Dataset & Tokenization


In [ ]:
corpus = [
    "I love  machine  learning",
    "I love deep learning",
    "I enjoy natural language processing",
    "Machine learning is fun",
    "Deep learning is powerful"
]
sentence = "I   love   machine learning ! !"

In [ ]:
# sentence tokenization 
def tokenize_sentence(sentence):
    puncs = ".?!,"
    sentence = sentence.lower()
    for punc in puncs:
        sentence = sentence.replace(punc,'')
    tokenized = sentence.split()

    return ["<s>"] + tokenized + ["</s>"]


In [ ]:
# corpus tokenization 
def tokenize_corpus(corpus: list[str]):
    tokenized = []    
    for sentence in corpus:        
        tokenized.append(tokenize_sentence(sentence))

    return tokenized 
    #  Or return [tokenize_sentence(sentence) for sentence in corpus]

In [ ]:
tokenized_corpus = tokenize_corpus(corpus)
tokenized_corpus

## Phase 2: vocabulary 

vocabulary is the set of all unique tokens that appear in the corpus.

In our implementation, we use Python's `set` because it automatically prevents duplicates entries.  

In [ ]:
def build_vocabulary (tokenized_corpus):
    vocab = set()

    for sentence in tokenized_corpus:
        for word in sentence:
            vocab.add(word)

    return vocab

## Phase 3: Unigram

In [ ]:
def count_unigrams(tokenized_corpus):
    count  = {}
    for sentence in tokenized_corpus:
        for word in sentence:
            if word in count:
                count[word] +=1
            else:
                count[word] = 1
    return (count)

In [ ]:
counts_unigrams = count_unigrams(tokenized_corpus)
print(counts_unigrams)

In [ ]:
def compute_unigram_probabilities(counted_unigrams:dict):
    total_words = sum(counted_unigrams.values())
    
    unigram_probabilities = {}
    
    for word, value in counted_unigrams.items():
        unigram_probabilities[word] = value/ total_words
    
    return (unigram_probabilities)

"""
Complexity

for vocabulary size V
we perform: sum(counted_unigrams.values()) 
which visits every unique word once.

So total complexity:
- Time: O(V)
- Space: O(V)
"""

## Phase 4: N-grams function 

**Sliding window**: algorithm technique  used to process consecutive elements efficiently.  

For N-grams:
- window size = n.
- -step = 1.

Sentence:
i love machine learning

n = 2

Windows:

(i, love)  
(love, machine)  
(machine, learning)  

**Formula**:

$$
\text {Number of n-grams=L−n+1}
$$

where 
- L: length of sentence.
- $
n \leq {L}
$


In [ ]:
def generate_ngrams(tokenized_corpus, n : int ):
    
    if n <= 0:
        raise ValueError("n must be greater than 0") 

    n_gram = []

    for sentence in tokenized_corpus:
        L = len(sentence)
        
        for i in range (L - n + 1):        

            n_gram.append(tuple(sentence [i:n+i]))
    
    return (n_gram)

""" 
Complexity:
- Time O(N)
- Space: O(number of generated n-grams)
"""


In [ ]:
generated_ngrams = generate_ngrams(tokenized_corpus,2)
print(generated_ngrams)

In [ ]:
def count_ngrams(generated_ngrams):
    count = {}
    for n_gram in generated_ngrams:
        if n_gram in count: # same as "if n_gram in count.keys()" dictionaries check their keys by default.
            count[n_gram] +=1
        else:
            count[n_gram] = 1 
    
    return count

""" 
Complexity:
- Time O(M) where M is the number of generated n-grams
- Space: O(U) where U is the number of unique n-grams
"""


In [ ]:
print(count_ngrams(generated_ngrams))

## Phase 5: Bigrams


In [ ]:
# Generate bigrams
bigrams = generate_ngrams(tokenized_corpus, 2)

In [ ]:
# Count bigrams
counts_bigram = count_ngrams(bigrams)

In [ ]:
def compute_bigram_probabilities(counts_bigram,counts_unigrams ):
    prob_bigram = {}
    
    for bigram,bigram_value in counts_bigram.items():
        unigram_value = counts_unigrams[bigram[0]]
        prob_bigram[bigram] = bigram_value/unigram_value
    return prob_bigram


In [ ]:
print (compute_bigram_probabilities(counts_bigram,counts_unigrams ))

## Phase 6: Sentence Scoring

Sentence Scoring can be used by both MLE and  laplace smoothed probabilities 

In [ ]:
sentence = ["i", "love", "machine", "learning"]

tokenized_corpus = tokenize_corpus(corpus)

counts_unigrams = count_unigrams(tokenized_corpus)

bigrams = generate_ngrams(tokenized_corpus, 2)

counts_bigram = count_ngrams(bigrams)

bigram_probabilities = compute_bigram_probabilities(counts_bigram,counts_unigrams )

print(bigram_probabilities)


In [ ]:
def compute_sentence_probability(tokenized_sentence,bigram_probabilities ):
    L = len(tokenized_sentence)
    prob = 1

    for i in range (L - 1):
        bigram = tuple(tokenized_sentence[i:2+i]) 
        if  bigram in bigram_probabilities:
            prob = prob * bigram_probabilities[bigram]
        else:
            return 0
    return prob 
# we wont use compute_sentence_probability instead we are going to use compute_sentence_log_probability

In [ ]:
sentence_probability = compute_sentence_probability(sentence,bigram_probabilities)
print(sentence_probability)

In [ ]:
def compute_sentence_log_probability(tokenized_sentence,bigram_probabilities ):
    L = len(tokenized_sentence)
    log_prob  = 0

    for i in range (L - 1):
        bigram = tuple(tokenized_sentence[i:2+i]) 
        if  bigram in bigram_probabilities:
            p = bigram_probabilities[bigram]
            log_prob  = log_prob  + np.log(p)
        else:
            return float("-inf") # we cant return 0 as log(0) = - inf so we use this: float("-inf")
    return log_prob

In [ ]:
sentence2 =  ["i", "love", "powerful", "learning"]

In [ ]:
sentence_probability = compute_sentence_probability(sentence2,bigram_probabilities)
print(sentence_probability)

In [ ]:
sentence_log_probability = compute_sentence_log_probability(sentence2,bigram_probabilities)
print(sentence_log_probability)

## Phase 7: Smoothing

In [ ]:
counts_unigrams = count_unigrams(tokenized_corpus)

counts_bigrams = count_ngrams(bigrams)

vocab = build_vocabulary(tokenized_corpus)
vocab_size = len(vocab) 


In [ ]:
k = 1
prob_bigram = {}
for bigram,bigram_value in counts_bigrams.items():
    unigram_value = counts_unigrams[bigram[0]]
    prob_bigram[bigram] = (bigram_value + k) /(unigram_value + k * vocab_size)
    print (bigram)

In [ ]:
def compute_bigram_probabilities_smoothed(counts_bigrams, counts_unigrams, vocab:set, k = 1 ):
    
    prob_bigram = {}
    vocab_size = len(vocab) 
    

    for previous_word in vocab:
        for current_word in vocab:
            bigram = (previous_word,current_word)
            unigram_value = counts_unigrams[previous_word]

            if bigram not in counts_bigrams: # the if condition is same as: bigram_value = counts_bigrams.get(bigram, 0) 
                bigram_value = 0
            else:
                bigram_value = counts_bigrams[bigram]
            
                
            prob_bigram[bigram] = (bigram_value + k) /(unigram_value + k * vocab_size)
    return (prob_bigram)


In [ ]:
bigram_probabilities_smoothed = compute_bigram_probabilities_smoothed(counts_bigrams,counts_unigrams,vocab )


**Should the probabilities sum to 1?**  

Yes but they don't all sum to 1.  
Instead for each previous word the conditional distribution sums to 1

$$
\sum_{w_i \in V} P(w_i \mid w_{i-1}) = 1
$$

Example:
``` text
Vocabulary:
<s>
I
love
NLP
</s>
```
then 

```text
P(<s> | love)
P(I | love)
P(love | love)
P(NLP | love)
P(</s> | love)
````
must sum to 1

In [ ]:
# test the sum of the probabilities for previous words
for previous_word in vocab:
    total = 0

    for current_word in vocab:
        total += bigram_probabilities_smoothed[(previous_word, current_word)]

    print(previous_word, total)

In [ ]:
# note the total sum is not equal 1
sum(bigram_probabilities_smoothed.values())

In [68]:
sentence2 =  "i love powerful learning"
sentence2 = tokenize_sentence(sentence2)
sentence2

['<s>', 'i', 'love', 'powerful', 'learning', '</s>']

```text
                Probability Model
                      │
        ┌─────────────┴─────────────┐
        │                           │
      MLE                     Laplace Smoothing
        │                           │
        └─────────────┬─────────────┘
                      │
compute_sentence_log_probability(...)
```

In [69]:
log_probability_smoothed = compute_sentence_log_probability(sentence2, bigram_probabilities_smoothed)
log_probability_smoothed 

np.float64(-10.565144066004702)